# Multi-Agent Financial Analysis System
**AAI-520 Final Team Project**  
**Tim Hays · Roopleen Kaur · Ashley Marx**  
Repository: [tim17hays/NLP_AAI520](https://github.com/tim17hays/NLP_AAI520)  
Evaluated code branch: [codex/final-team-integration](https://github.com/tim17hays/NLP_AAI520/tree/codex/final-team-integration)  
Submission source: `notebooks/final_multi_agent_financial_analysis.ipynb`

This notebook combines retrieval/news, planning/routing, and evaluation/revision/memory. It distinguishes provider snapshots, synthetic offline checks, and real-model calls on fixed fictional evidence. The final analyses are research demonstrations, not investment recommendations.

**Reading the results:** Ashley's demonstrations show that a lesson was saved after Run 1 and retrieved before Run 2. The recorded runs did not demonstrate improved analysis quality. Both controlled final analyses failed acceptance; those failures remain visible.

**Execution:** In Colab, choose **Runtime → Change runtime type → T4 GPU**, then **Run all** in a fresh runtime. Colab enables actual model calls by default. Local validation uses clearly labeled recorded evidence and offline checks, and does not claim a new GPU run. Save the executed notebook before exporting/submitting.

## 1. Design, ownership and rubric coverage
| Requirement | Implementation | Observable evidence in this notebook |
|---|---|---|
| Plans research | `ResearchPlanner.plan()` | Research steps, selected specialists, lesson-review step |
| Uses tools dynamically | Planner calls selected market/financial/news tools | Route decisions, source-attributed payloads and tool/error trace |
| Self-reflects | Qwen evaluation plus citation/benchmark checks | Scores, critical issues, feedback and acceptance decision |
| Learns across runs | Process-lesson JSON memory | Run 1 lesson record, disk persistence, retrieval before Run 2 and delivery to drafting |
| Prompt chaining | Ingest → preprocess → classify → extract → summarize | Intermediate news trace and provenance checks |
| Routing | Market, earnings, news and company specialists | Three specialist examples and fallback behavior |
| Evaluator–optimizer | Draft → evaluate → bounded revision | Initial/final drafts, revision histories and stopping rules |

Tim owns retrieval/news and integration; Roopleen owns planning/routing and draft integration; Ashley owns evaluation, revision and memory. Their original notebooks and Git history remain in the repository. This assembly reuses their code rather than replacing their implementations.

The planner's routing is keyword based. Model-based critique, generation and revision are probabilistic. Storing and delivering a process lesson demonstrates memory use; it does not establish improved factual quality or a causal memory effect.

In [1]:
from IPython.display import HTML, Markdown, display
stages = ["Plan / route", "Retrieve / news chain", "Draft", "Evaluate / revise", "Save lesson → next plan"]
boxes = ''.join(f'<div style="flex:1;padding:14px 8px;border:1px solid #8caac5;border-radius:8px;background:#edf4fb;text-align:center">{s}</div>' for s in stages)
display(HTML('<div style="display:flex;gap:8px;flex-wrap:wrap;font:14px system-ui">'+boxes+'</div>'))

## 2. Reproducible runtime and common repository
Qwen uses `Qwen/Qwen2.5-1.5B-Instruct`; FinBERT uses `ProsusAI/finbert`. Models load through Hugging Face Transformers. No paid LLM API key is needed. Optional Finnhub/Alpha Vantage keys are data-provider credentials and must remain private.

The Colab setup creates an isolated clone, checks out the fetched integration revision and prints its exact commit. The local audit uses the existing checkout. Existing edits in a reused Colab clone cause a clear stop. Model weights download on first use; no Google Drive access is required. Download the generated ZIP before the Colab runtime ends.

In [2]:
import importlib.util, json, os, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone
IN_COLAB = bool(importlib.util.find_spec("google")) and bool(importlib.util.find_spec("google.colab"))
REPO_URL = "https://github.com/tim17hays/NLP_AAI520.git"
TEAM_BRANCH = "codex/final-team-integration"
RUN_LIVE_MODELS = IN_COLAB and os.getenv("AAI520_RECORDED_ONLY") != "1"
if IN_COLAB:
    REPO = Path("/content/NLP_AAI520-final-review")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--branch", TEAM_BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        changes = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True)
        if changes.strip():
            raise RuntimeError("Existing Colab clone has edits. Preserve them before reusing this folder.")
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", TEAM_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
else:
    REPO = Path.cwd()
    if REPO.name == "notebooks": REPO = REPO.parent
if not (REPO / "src/evaluation_memory.py").is_file():
    raise RuntimeError("Run from the combined project checkout.")
os.chdir(REPO)
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
TEAM_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Code commit:", TEAM_COMMIT)
print("Python:", sys.version.split()[0])
print("Execution mode:", "fresh Colab model calls" if RUN_LIVE_MODELS else "recorded-evidence audit; no new model inference")
if RUN_LIVE_MODELS:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Colab CUDA GPU runtime, then restart Run all.")
    print("GPU:", torch.cuda.get_device_name(0))
    print("Torch:", torch.__version__)

Code commit: 37f985826bdd388d473e3aa280c9c8e4b292fa06
Python: 3.12.14
Execution mode: recorded-evidence audit; no new model inference


In [3]:
import inspect
from dataclasses import asdict, is_dataclass
from src.data_tools import StockResearchTools
from src.news_pipeline import NewsPipeline
from src.planning import ResearchPlanner
from src.llm import MODEL_NAME, generate_draft
from src import evaluation_memory as evaluation
# Reuse loaded model weights but refresh the small response-handling modules.
import importlib
from src import model_transport
importlib.reload(model_transport)
importlib.reload(evaluation)
from src.evaluation_memory import (
    qwen_call_model, run_research_with_memory, MODEL_CALL_LOG, MAX_REVISIONS,
    check_citations, derived_calculations, read_lessons, utc_now, json_copy, require_text,
)
TEAM_PLANNER = ResearchPlanner()
CALL_MODEL = qwen_call_model
OUTPUT_DIR = REPO / "outputs/final_submission"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Draft model:", MODEL_NAME)
assert getattr(qwen_call_model, "schema_attempt_limit", None) == 2, "Refresh the repository/imports before running models."
print("Structured response attempts per call:", qwen_call_model.schema_attempt_limit)
print("Planner signature:", inspect.signature(ResearchPlanner.execute))
print("Revision limit:", MAX_REVISIONS)

Draft model: Qwen/Qwen2.5-1.5B-Instruct
Structured response attempts per call: 2
Planner signature: (self, ticker: 'str', research_question: 'str', lessons: 'list | None' = None) -> 'dict[str, Any]'
Revision limit: 2


## 3. News prompt chain and source validation — Tim
The production chain ingests provider news, normalizes text and URLs, deduplicates articles, classifies available text using FinBERT, extracts source sentences, and produces an extractive summary. It records timestamps and errors. The trace below is a **synthetic offline test with a deterministic injected classifier**, not a FinBERT accuracy measurement.

Yahoo news requests have a 20-second per-request timeout; successful results are cached for 15 minutes. The fallback order is Yahoo → Finnhub → Alpha Vantage. Historical-price requests use a 15-second timeout. Those are per-request settings, not an overall workflow deadline. News headlines/short summaries are preliminary evidence; a sentiment label is neither a verified event nor a forecast.

In [4]:
from pathlib import Path
import sys, json
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from src.news_pipeline import NewsPipeline
from src.data_tools import StockResearchTools
class FixtureClassifier:
    def classify(self, text):
        return "neutral", 1.0
raw = [
    {"title": "Apple synthetic financial report", "summary": "Revenue was 100 in the synthetic example.",
     "publisher": "Synthetic fixture", "link": "https://example.com/aapl-fixture?tracking=1"},
    {"title": "Duplicate fixture", "summary": "This duplicate should be removed.",
     "publisher": "Synthetic fixture", "link": "https://example.com/aapl-fixture"},
]
result = NewsPipeline(classifier=FixtureClassifier()).process("AAPL", raw)
print(json.dumps(result["trace"], indent=2))

{
  "ingest": {
    "input_count": 2
  },
  "preprocess": {
    "retained_count": 1,
    "records": [
      {
        "text": "Apple synthetic financial report. Revenue was 100 in the synthetic example.",
        "url": "https://example.com/aapl-fixture"
      }
    ]
  },
  "classify_finbert": [
    {
      "url": "https://example.com/aapl-fixture",
      "label": "neutral",
      "score": 1.0
    }
  ],
  "extract": [
    {
      "url": "https://example.com/aapl-fixture",
      "facts": [
        "Revenue was 100 in the synthetic example."
      ]
    }
  ],
  "summarize": "Processed 1 source-attributed news items (1 neutral). Representative headlines: Apple synthetic financial report."
}


In [5]:
assert len(result["items"]) == 1
item = result["items"][0]
assert item["source"]["url"] == "https://example.com/aapl-fixture"
assert all(fact in result["trace"]["preprocess"]["records"][0]["text"] for fact in item["extracted_facts"])
assert item["title"] in result["summary"]
print("PASS: deduplication, extracted-source alignment, headline-summary alignment")
print(result["summary"])

PASS: deduplication, extracted-source alignment, headline-summary alignment
Processed 1 source-attributed news items (1 neutral). Representative headlines: Apple synthetic financial report.


In [6]:
class FailedTicker:
    def get_news(self, **kwargs):
        raise TimeoutError("synthetic provider timeout")
empty = NewsPipeline(ticker_factory=lambda _: FailedTicker(), classifier=FixtureClassifier(), fallback_news_sources=[]).process("AAPL")
assert empty["items"] == [] and empty["errors"]
print(json.dumps({"summary": empty["summary"], "errors": empty["errors"]}, indent=2))
class FailedClassifier:
    def classify(self, text):
        raise RuntimeError("synthetic model unavailable")
failed = NewsPipeline(classifier=FailedClassifier()).process("AAPL", raw[:1])
assert failed["items"][0]["sentiment"] == "unavailable"
print(json.dumps({"sentiment": failed["items"][0]["sentiment"], "errors": failed["errors"]}, indent=2))

{
  "summary": "No deduplicated news evidence was available for this run.",
  "errors": [
    "Yahoo Finance news failed: synthetic provider timeout"
  ]
}
{
  "sentiment": "unavailable",
  "errors": [
    "FinBERT classification failed for 'Apple synthetic financial report': synthetic model unavailable"
  ]
}


The checks link extracted sentences to the retained source text, confirm deduplication, and preserve missing-data/classifier failures. They verify provenance and error behavior, not truth of the article or investment value. Financial snapshots do not establish a reporting period unless the provider explicitly supplies one.

## 4. Planning, specialist routing and draft generation — Roopleen
The planner selects specialists using research-question keywords, calls the corresponding tools, and returns evidence, a structured plan, tool/error trace, retrieved lessons and a draft. Unsupported or unclear wording falls back to a company-profile route; this is a documented heuristic rather than evidence of semantic understanding.

In [7]:
questions = {
    "market": "Summarize stock price performance.",
    "earnings": "Summarize earnings and revenue.",
    "news": "Summarize recent news sentiment.",
    "fallback": "Explain this unrelated request.",
}
for name, question in questions.items():
    plan = TEAM_PLANNER.plan("AAPL", question)
    print(name, "→", plan.specialists)
lesson_plan = TEAM_PLANNER.plan("AAPL", "market and financial news", lessons=["Do not infer reporting periods."])
print("Lesson-aware first step:", lesson_plan.steps[0])
print("\nPlanner execution code:\n", inspect.getsource(ResearchPlanner.execute))

market → ['market']
earnings → ['earnings']
news → ['news']
fallback → ['company']
Lesson-aware first step: Review retrieved lessons before collecting and drafting evidence: ['Do not infer reporting periods.']



Planner execution code:
     def execute(self, ticker: str, research_question: str, lessons: list | None = None) -> dict[str, Any]:
        """Execute the specialists selected for the research question."""
        symbol = self.normalize_ticker(ticker)
        specialists = self.route(research_question)

        if not specialists:
            specialists = ["company"]

        plan = self.plan(symbol, research_question, lessons=lessons)
        evidence: dict[str, Any] = {}

        for specialist in specialists:
            if specialist == "market":
                result = self._research_tools.market_snapshot(symbol)
                evidence["market"] = result.to_dict()

            elif specialist == "earnings":
                result = self._research_tools.financial_snapshot(symbol)
                evidence["earnings"] = result.to_dict()

            elif specialist == "company":
                result = self._research_tools.company_profile(symbol)
                evidence["comp

### AAPL provider evidence and Qwen draft
In Colab this cell retrieves a new AAPL snapshot and generates a new Qwen draft. In local recorded-evidence mode it loads Roopleen's saved AAPL handoff and preserves its provider access dates. This snapshot is educational research and can change between executions. Keep missing values and source errors visible.

In [8]:
if RUN_LIVE_MODELS:
    AAPL_RESULT = TEAM_PLANNER.execute(
        "AAPL", "Summarize Apple's market performance, financial metrics, and recent news with source URLs and limitations.",
        lessons=[],
    )
    AAPL_MODE = "fresh provider/model execution"
else:
    AAPL_RESULT = json.loads((REPO / "notebooks/person2_aapl_live_handoff.json").read_text())
    AAPL_MODE = "Roopleen's recorded provider/model execution; not refreshed locally"
print("AAPL mode:", AAPL_MODE)
print("Plan:", json.dumps(AAPL_RESULT["plan"], indent=2))
print("Tool/error trace:", json.dumps(AAPL_RESULT["tool_trace"], indent=2))
for name in ("market", "earnings"):
    print(name, json.dumps(AAPL_RESULT["evidence"].get(name, {}), indent=2))
news = AAPL_RESULT["evidence"].get("news", {})
print("News sources:", json.dumps([{"title": i.get("title"), "url": i.get("url"), "sentiment": i.get("sentiment")} for i in news.get("items", [])], indent=2))
print("DRAFT (model output; inspect against evidence):\n", AAPL_RESULT["draft_analysis"])
(OUTPUT_DIR / "aapl_result.json").write_text(json.dumps(AAPL_RESULT, indent=2))

AAPL mode: Roopleen's recorded provider/model execution; not refreshed locally
Plan: {
  "ticker": "AAPL",
  "research_question": "Summarize Apple's market performance, financial metrics, and recent news.",
  "specialists": [
    "market",
    "earnings",
    "news"
  ],
  "steps": [
    "Review retrieved lessons before collecting and drafting evidence: ['Do not invent currency, financial units, or reporting periods.', 'Cite source URLs and distinguish headlines from verified facts.']",
    "Retrieve evidence from the market specialist.",
    "Retrieve evidence from the earnings specialist.",
    "Retrieve evidence from the news specialist.",
    "Combine the retrieved evidence into a draft analysis.",
    "Send the draft analysis and supporting evidence for evaluation."
  ],
  "retrieved_lessons": [
    "Do not invent currency, financial units, or reporting periods.",
    "Cite source URLs and distinguish headlines from verified facts."
  ]
}
Tool/error trace: [
  {
    "specialist": 

20638

The recorded AAPL draft omitted individual source URLs and included news that was not directly about AAPL. Source-attributed retrieval alone does not guarantee a grounded generated answer. Inspect any fresh draft for the same issues; do not treat a generated narrative or a high model score as independent verification.

## 5. Evaluation, revision and persistent memory — Ashley
The evaluator scores evidence support, completeness, consistency, uncertainty and clarity from 0–2. Passing requires at least 8/10, no zero criterion, full evidence support and consistency, and no critical issues. Citation checks reject missing/unknown references. Narrow benchmark checks validate selected computed facts; they are not general factual verification.

The loop stops on acceptance, at two revisions, or when a revised draft is unchanged. It never converts a failure into a pass for presentation. Process lessons can be saved after a verified failure without endorsing the failed financial claims. Duplicate lessons reuse an existing record; `saved_lesson` being non-null does not mean a second unique lesson was created.

The shared transport permits one additional model attempt when JSON/schema validation fails. The retry receives the original evidence plus the exact validation error and raw invalid response. Both attempts remain logged. Missing criteria are never supplied by Python, and two invalid attempts still stop the run. This format retry does not change scoring or count as an analysis revision.


In [9]:
print("Evaluation and guard checks:\n", inspect.getsource(evaluation.evaluate_analysis))
print("Bounded revision:\n", inspect.getsource(evaluation.run_evaluation_revision))
print("Process-lesson policy:\n", inspect.getsource(evaluation.select_process_lesson))

Evaluation and guard checks:
 def evaluate_analysis(context, draft, call_model, lessons=None):
    payload = build_evaluation_payload(
        context,
        draft,
        lessons,
    )

    check = check_citations(
        payload["draft_analysis"],
        payload["evidence"],
    )
    payload["deterministic_citation_check"] = check

    content_check = check_benchmark_content(context, draft)
    payload["deterministic_content_check"] = content_check
    payload["derived_calculations"] = derived_calculations(payload["evidence"])

    raw = validate_evaluation(
        call_json_model(
            call_model,
            EVALUATOR_INSTRUCTIONS,
            payload,
        )
    )

    evaluation = json_copy(raw)
    lesson_origin = "model" if raw["lesson"].strip() else "none"

    if check["issues"]:
        cap = 1 if check["recognized_references"] else 0

        evaluation["scores"]["evidence_support"] = min(
            evaluation["scores"]["evidence_support"],
            c

### Controlled benchmark and offline checks
The DEMO company, figures, dates, news and `example.com` URLs are fictional classroom evidence. Expected calculations are 20% revenue growth, 10% operating margin and 10% closing-price change. No prior operating-income value, guaranteed return or price target is supplied.

The acceptance checks below use scripted fixtures; this verifies code/data flow and stopping rules, not real model performance. The following section separately uses real Qwen responses.

In [10]:
benchmark = json.loads((REPO / "data/examples/controlled_demo_inputs.json").read_text())
DEMO_QUESTION = benchmark["DEMO_QUESTION"]
SAMPLE_CONTEXT = benchmark["SAMPLE_CONTEXT"]
REFERENCE_ANALYSIS = benchmark["REFERENCE_ANALYSIS"]
print("Fictional evidence:", json.dumps(SAMPLE_CONTEXT["evidence"], indent=2))
print("Derived benchmark calculations:", derived_calculations(SAMPLE_CONTEXT["evidence"]))
checks = subprocess.run([sys.executable, "-m", "pytest", "-q"], text=True, capture_output=True,
                        env={**os.environ, "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "AAI520_LIVE_TESTS": "0"})
print(checks.stdout)
if checks.returncode:
    print(checks.stderr)
    raise RuntimeError("Offline checks failed; resolve before submission.")

Fictional evidence: {
  "market": {
    "tool": "market_snapshot",
    "ticker": "DEMO",
    "data": {
      "currency": "USD",
      "first_close": 40,
      "latest_close": 44,
      "start_date": "2026-08-31",
      "as_of": "2026-09-30",
      "dividends_and_corporate_actions": "not supplied"
    },
    "sources": [
      {
        "name": "Fictional closing-price record",
        "url": "https://example.com/fictional-demo/prices",
        "accessed_at": "2026-10-01T00:00:00+00:00"
      }
    ],
    "errors": []
  },
  "earnings": {
    "tool": "financial_snapshot",
    "ticker": "DEMO",
    "data": {
      "currency": "USD",
      "unit": "million",
      "current_period": "Q2 2026",
      "previous_period": "Q2 2025",
      "current_revenue": 120,
      "previous_revenue": 100,
      "operating_income": 12,
      "management_forecast": null
    },
    "sources": [
      {
        "name": "Fictional classroom financial statement",
        "url": "https://example.com/fictional-dem

..............................ss......                                   [100%]
36 passed, 2 skipped in 0.06s



## 6. Real-model two-run memory demonstration
Run 1 starts with a human-written benchmark draft whose citations are deliberately removed. Run 2 uses the team's Qwen generator with the lesson retrieved from disk. Both runs use the same fixed fictional evidence; evaluator and revision calls are real Qwen calls in Colab.

The differing initial draft origins and changing model outputs prevent a causal claim that memory improves quality. Delivery, final acceptance, and score change are reported separately. On local audit, the exact authored code is shown but its model calls are skipped; Section 7 loads Ashley's recorded results instead.

In [11]:
import copy
import json

from dataclasses import asdict, is_dataclass
from pathlib import Path
from uuid import uuid4
from zipfile import ZIP_DEFLATED, ZipFile

required_names = [
    "TEAM_PLANNER",
    "CALL_MODEL",
    "qwen_call_model",
    "MODEL_NAME",
    "TEAM_COMMIT",
    "REFERENCE_ANALYSIS",
    "SAMPLE_CONTEXT",
    "DEMO_QUESTION",
    "MODEL_CALL_LOG",
    "OUTPUT_DIR",
    "MAX_REVISIONS",
    "run_research_with_memory",
    "check_citations",
    "json_copy",
    "require_text",
    "utc_now",
]

missing_names = [
    name for name in required_names if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Run the earlier notebook cells through Section 11 first. "
        "Missing: " + ", ".join(missing_names)
    )

if CALL_MODEL is not qwen_call_model:
    raise RuntimeError(
        "Section 11 must connect CALL_MODEL to the real Qwen model."
    )

from src.llm import generate_draft


def remove_reference_citations(text):
    """Prepare a deliberately uncited, human-written test draft."""
    for citation in (
        "[evidence.earnings; evidence.news.items[0]]",
        "[evidence.news.items[0]]",
        "[evidence.earnings]",
        "[evidence.market]",
    ):
        text = text.replace(citation, "")

    return text.strip()


CONTROLLED_START_DRAFT = remove_reference_citations(
    REFERENCE_ANALYSIS
)

assert not check_citations(
    CONTROLLED_START_DRAFT,
    SAMPLE_CONTEXT["evidence"],
)["passed"]


class FrozenEvidencePlanner:
    """Use team planning and drafting with frozen fictional evidence."""

    def __init__(
        self,
        team_planner,
        draft_generator,
        initial_draft=None,
    ):
        self.team_planner = team_planner
        self.draft_generator = draft_generator
        self.initial_draft = initial_draft
        self.calls = []

    def execute(self, ticker, research_question, lessons=None):
        lessons = json_copy(lessons or [])
        context = copy.deepcopy(SAMPLE_CONTEXT)
        context["verification_profile"] = "frozen_demo_v1"

        if ticker != context["ticker"]:
            raise ValueError(
                "This controlled test only accepts DEMO."
            )

        plan = self.team_planner.plan(
            ticker,
            research_question,
            lessons=json_copy(lessons),
        )

        plan = (
            asdict(plan)
            if is_dataclass(plan)
            else json_copy(plan)
        )

        context.update(
            {
                "research_question": research_question,
                "plan": plan,
                "retrieved_lessons": lessons,
                "tool_trace": [
                    {
                        "specialist": name,
                        "tool": "frozen_fictional_evidence",
                        "errors": item.get("errors", []),
                    }
                    for name, item in context["evidence"].items()
                ],
            }
        )

        record = {
            "created_at": utc_now(),
            "lessons_to_plan": json_copy(lessons),
            "plan": json_copy(plan),
            "status": "started",
            "draft_source": (
                "human-written controlled input"
                if self.initial_draft is not None
                else "team Qwen generate_draft"
            ),
            "lessons_to_generator": [],
        }

        self.calls.append(record)

        try:
            if self.initial_draft is not None:
                draft = self.initial_draft
            else:
                record["lessons_to_generator"] = json_copy(
                    lessons
                )

                draft = self.draft_generator(
                    ticker=ticker,
                    research_question=research_question,
                    evidence=json_copy(context["evidence"]),
                    lessons=json_copy(lessons),
                )

            context["draft_analysis"] = require_text(
                draft,
                "draft_analysis",
            )

            record["draft_analysis"] = draft
            record["status"] = "completed"

            return context

        except Exception as exc:
            record["status"] = "error"
            record["error"] = str(exc)
            raise


def write_json(path, value):
    Path(path).write_text(
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        ),
        encoding="utf-8",
    )


def archive_experiment(folder):
    archive = folder.with_suffix(".zip")

    with ZipFile(archive, "w", ZIP_DEFLATED) as bundle:
        for path in sorted(folder.iterdir()):
            if path.is_file():
                bundle.write(
                    path,
                    arcname=folder.name + "/" + path.name,
                )

    return archive


def run_controlled_demonstration(
    team_planner,
    draft_generator,
    model_call,
):
    """Keep workflow delivery and analysis quality separate."""
    experiment_id = str(uuid4())

    folder = Path(OUTPUT_DIR) / (
        "controlled_demo_" + experiment_id
    )
    folder.mkdir(parents=True, exist_ok=False)

    memory_file = folder / "controlled_lessons.json"
    write_json(memory_file, [])

    first = FrozenEvidencePlanner(
        team_planner,
        draft_generator,
        initial_draft=CONTROLLED_START_DRAFT,
    )

    second = FrozenEvidencePlanner(
        team_planner,
        draft_generator,
    )

    experiment = {
        "experiment_id": experiment_id,
        "folder": str(folder),
        "results": [],
        "workflow_checks": {},
        "quality_checks": {},
        "execution_error": None,
    }

    globals()["CONTROLLED_EXPERIMENT"] = experiment
    call_start = len(MODEL_CALL_LOG)

    write_json(
        folder / "test_conditions.json",
        {
            "experiment_id": experiment_id,
            "model": MODEL_NAME,
            "repository_commit": TEAM_COMMIT,
            "evidence_mode": (
                "fixed fictional DEMO evidence; no live retrieval"
            ),
            "run_1_initial_draft": (
                "human-written; citations deliberately removed"
            ),
            "run_2_initial_draft": (
                "team Qwen generator with retrieved lessons"
            ),
            "max_revisions": MAX_REVISIONS,
            "memory_policy": (
                "process lessons from success or verified failure"
            ),
            "benchmark_checks": (
                "minimum values, missing-data claims, prior income"
            ),
            "not_claimed": (
                "causal proof that memory improves model quality"
            ),
        },
    )

    write_json(
        folder / "controlled_input.json",
        {
            **SAMPLE_CONTEXT,
            "verification_profile": "frozen_demo_v1",
            "draft_analysis": CONTROLLED_START_DRAFT,
            "derived_calculations": derived_calculations(
                SAMPLE_CONTEXT["evidence"]
            ),
        },
    )

    try:
        for number, planner in enumerate((first, second), 1):
            print(
                f"\nStarting controlled real-model run {number}..."
            )

            result = run_research_with_memory(
                planner,
                "DEMO",
                DEMO_QUESTION,
                model_call,
                memory_path=memory_file,
            )

            result["execution_mode"] = (
                "real_model_fictional_evidence"
            )
            experiment["results"].append(result)

            write_json(
                folder / f"controlled_run_{number}.json",
                result,
            )

            print("\nINITIAL DRAFT\n", result["initial_draft"])
            print("\nFINAL DRAFT\n", result["revised_analysis"])

            print(
                "\nEVALUATION HISTORY\n",
                json.dumps(
                    result["revision_history"],
                    indent=2,
                ),
            )

            print(
                "\nSAVED LESSON\n",
                json.dumps(result["saved_lesson"], indent=2),
            )

            print(
                "LESSON ORIGIN:",
                result["saved_lesson_origin"],
            )

        run1, run2 = experiment["results"]
        source_id = run1["run_id"]

        def from_first(records):
            return any(
                isinstance(record, dict)
                and record.get("source_run_id") == source_id
                for record in records
            )

        workflow = {
            "run_1_initial_failure_detected": (
                not run1["revision_history"][0]["passed"]
            ),
            "run_1_revision_attempted": (
                run1["revision_calls"] > 0
            ),
            "run_1_changed_draft_evaluated": (
                run1["revisions_completed"] > 0
            ),
            "run_1_process_lesson_saved": (
                run1["saved_lesson"] is not None
            ),
            "run_1_lesson_read_from_disk": from_first(
                read_lessons(memory_file)
            ),
            "run_2_retrieved_run_1_lesson": from_first(
                run2["lessons_used"]
            ),
            "run_2_plan_received_run_1_lesson": from_first(
                run2["plan"].get("retrieved_lessons", [])
            ),
            "run_2_generator_received_run_1_lesson": from_first(
                second.calls[-1]["lessons_to_generator"]
            ),
        }

        workflow["all_workflow_checks_passed"] = all(
            workflow.values()
        )

        quality = {
            "run_1_final_passed": run1["passed"],
            "run_2_final_passed": run2["passed"],
            "run_1_score_increased": (
                run1["revised_score"] > run1["initial_score"]
            ),
            "run_2_score_increased": (
                run2["revised_score"] > run2["initial_score"]
            ),
            "both_final_analyses_passed": (
                run1["passed"] and run2["passed"]
            ),
        }

        experiment["workflow_checks"] = workflow
        experiment["quality_checks"] = quality

        print(
            "\nWORKFLOW / MEMORY CHECKS\n",
            json.dumps(workflow, indent=2),
        )

        print(
            "\nANALYSIS QUALITY CHECKS\n",
            json.dumps(quality, indent=2),
        )

    except Exception as exc:
        experiment["execution_error"] = {
            "type": type(exc).__name__,
            "message": str(exc),
        }

        print(
            "Execution stopped. Available logs are preserved; "
            "results are partial."
        )
        raise

    finally:
        experiment["finished_at"] = utc_now()

        write_json(
            folder / "model_calls.json",
            MODEL_CALL_LOG[call_start:],
        )

        write_json(
            folder / "planner_and_draft_calls.json",
            first.calls + second.calls,
        )

        write_json(
            folder / "execution_status.json",
            {
                key: value
                for key, value in experiment.items()
                if key != "results"
            },
        )

        (folder / "manual_review.txt").write_text(
            "Compare the generated drafts with controlled_input.json.\n"
            "Revenue growth 20%; operating margin 10%; "
            "price change 10%.\n"
            "No prior-year operating-income number was supplied.\n"
            "No guaranteed return, forecast or price target "
            "was supplied.\n"
            "Supplier-delay impact is unquantified.\n"
            "Inspect citation support and each lesson's provenance.\n"
            "Record remaining errors in notebook Section 14.\n",
            encoding="utf-8",
        )

        archive = archive_experiment(folder)
        globals()["CONTROLLED_ZIP"] = archive

        print("\nRESULTS FOLDER:", folder)
        print("RESULTS ZIP:", archive)

    return experiment


for name in ("CONTROLLED_EXPERIMENT", "CONTROLLED_ZIP"):
    globals().pop(name, None)

if RUN_LIVE_MODELS:
    CONTROLLED_EXPERIMENT = run_controlled_demonstration(
        TEAM_PLANNER, generate_draft, CALL_MODEL,
    )
else:
    print("No fresh model execution in local recorded-evidence mode.")

print(
    "Run Section 14 to display the observed results "
    "and update the ZIP."
)

No fresh model execution in local recorded-evidence mode.
Run Section 14 to display the observed results and update the ZIP.


## 7. Observed results, memory delivery and manual review
The view below uses fresh results after a Colab run, or the recorded stdout from Ashley's authored demonstration in local audit mode. Each source is labeled. Values in the historical description remain labeled as historical rather than being rewritten to predict a future execution.

In [12]:
import html
if RUN_LIVE_MODELS:
    observed = CONTROLLED_EXPERIMENT
    observed_runs = observed["results"]
    evidence_label = "FRESH COLAB MODEL EXECUTION"
else:
    observed = json.loads((REPO / "data/examples/recorded_controlled_results.json").read_text())
    observed_runs = observed["results"]
    evidence_label = "ASHLEY'S RECORDED REAL-MODEL EXECUTION; NOT A NEW LOCAL GPU RUN"
print(evidence_label)
rows = []
for number, r in enumerate(observed_runs, 1):
    lessons_received = len(r["lessons_used"]) if "lessons_used" in r else (0 if number == 1 else int(observed["workflow_checks"]["run_2_retrieved_run_1_lesson"]))
    rows.append(f"<tr><td>{number}</td><td>{r['initial_score']}/10</td><td>{r['revised_score']}/10</td><td>{r['revisions_completed']}</td><td>{r['passed']}</td><td>{lessons_received}</td></tr>")
display(HTML('<table><thead><tr><th>Run</th><th>Initial</th><th>Final</th><th>Revisions</th><th>Accepted</th><th>Lessons received</th></tr></thead><tbody>'+''.join(rows)+'</tbody></table>'))
print("Workflow / memory checks:", json.dumps(observed["workflow_checks"], indent=2))
print("Quality checks:", json.dumps(observed["quality_checks"], indent=2))
for number, r in enumerate(observed_runs, 1):
    print("\nRUN", number, "INITIAL DRAFT\n", r["initial_draft"])
    print("FINAL DRAFT\n", r["revised_analysis"])
    print("EVALUATION HISTORY\n", json.dumps(r["revision_history"], indent=2))
    print("SAVED LESSON\n", json.dumps(r["saved_lesson"], indent=2))
unique_lessons = {r["saved_lesson"]["lesson_id"] for r in observed_runs if r["saved_lesson"] is not None}
print("Unique lesson records returned:", len(unique_lessons))
print("A returned saved lesson may be an existing deduplicated record.")

ASHLEY'S RECORDED REAL-MODEL EXECUTION; NOT A NEW LOCAL GPU RUN


Run,Initial,Final,Revisions,Accepted,Lessons received
1,7/10,7/10,2,False,0
2,5/10,5/10,1,False,1


Workflow / memory checks: {
  "run_1_initial_failure_detected": true,
  "run_1_revision_attempted": true,
  "run_1_changed_draft_evaluated": true,
  "run_1_process_lesson_saved": true,
  "run_1_lesson_read_from_disk": true,
  "run_2_retrieved_run_1_lesson": true,
  "run_2_plan_received_run_1_lesson": true,
  "run_2_generator_received_run_1_lesson": true,
  "all_workflow_checks_passed": true
}
Quality checks: {
  "run_1_final_passed": false,
  "run_2_final_passed": false,
  "run_1_score_increased": false,
  "run_2_score_increased": false,
  "both_final_analyses_passed": false
}

RUN 1 INITIAL DRAFT
 DEMO's Q2 2026 revenue was USD 120 million versus USD 100 million in Q2 2025,
a 20% increase: (120 - 100) / 100 * 100. Operating income of USD 12 million
implies an operating margin of 10%: 12 / 120 * 100. 

The closing price rose from USD 40 on August 31 to USD 44 on September 30,
2026, a 10% price change. This is not an adjusted total-return calculation:
dividend and corporate-action infor

### Historical interpretation and limitations
In Ashley's recorded demonstration, Run 1 stayed at 7/10 after two revisions and stopped at the revision limit. Run 2 stayed at 5/10 after one completed revision and stopped when the next draft was unchanged. Neither final analysis passed. All recorded workflow/memory-delivery checks passed, including retrieval of Run 1's process lesson before Run 2 planning and generation.

The lesson was delivered but the generated output still omitted valid citations. Run 2 misstated 20% revenue growth as 10%, confused the $4 closing-price increase with a 4% return (the correct change is 10%), and invented a prior 12% operating margin. Run 1 also introduced an unsupported prior-quarter comparison and operating-efficiency trend. Some were not caught by the model's scores; manual review remained necessary.

These results demonstrate an implemented feedback workflow and memory delivery, **not improved financial analysis quality**. The synthetic offline reference example demonstrates successful logic under scripted responses, not a successful real-model outcome. A paired with/without-memory experiment using the same initial conditions would be needed to isolate memory's effect.

For any fresh Colab execution, compare the displayed drafts to the fixed evidence and record that execution's actual issues. Do not carry the historical score numbers forward as if newly measured.

In [13]:
manual_findings = []
for number, r in enumerate(observed_runs, 1):
    citation_check = evaluation.check_citations(r["revised_analysis"], SAMPLE_CONTEXT["evidence"])
    benchmark_check = evaluation.check_benchmark_content({**SAMPLE_CONTEXT, "verification_profile": "frozen_demo_v1"}, r["revised_analysis"])
    finding = {"run": number, "citation_check": citation_check, "benchmark_check": benchmark_check}
    manual_findings.append(finding)
    print(json.dumps(finding, indent=2))
print("These narrow rule checks do not replace a claim-by-claim manual review.")

{
  "run": 1,
  "citation_check": {
    "recognized_references": [],
    "unknown_references": [],
    "issues": [
      "No valid supplied citation was found. Cite the supplied source URLs or evidence paths alongside the factual claims."
    ],
    "passed": false,
    "scope": "Minimum citation presence and reference validity only."
  },
  "benchmark_check": {
    "applied": true,
    "checks": [
      "Minimum coverage of computed percentage values.",
      "Flag missing-data claims about available benchmark metrics.",
      "Flag the observed unsupported prior-income comparison."
    ],
    "issues": [],
    "passed": true,
    "scope": "Narrow benchmark checks. Passing is not proof that every claim, calculation association, or citation is correct."
  }
}
{
  "run": 2,
  "citation_check": {
    "recognized_references": [],
    "unknown_references": [],
    "issues": [
      "No valid supplied citation was found. Cite the supplied source URLs or evidence paths alongside the factual 

## 8. Submission artifact, reproducibility and final checks
This notebook visibly includes all four functions and three workflow patterns, with the limits of the recorded real-model results. Model-generated financial claims are never treated as verified merely because a workflow completed.

After Colab **Run all**, save/download the executed `.ipynb`. The final cell exports the in-memory executed notebook to HTML and downloads an artifact ZIP when Colab's notebook-message API is available. If that API is unavailable, it saves a clear export status and you can export the downloaded notebook locally with `python scripts/export_submission.py PATH_TO_EXECUTED_NOTEBOOK`. Review code wrapping, tables, source links, outputs and disclosure in the final HTML before Canvas upload. HTML is an allowed final format; PDF is preferred by the assignment.

The individual peer evaluations, team approval and Canvas upload confirmation are human submission steps and cannot be inferred from notebook execution.

## References and evidence provenance
- Qwen Team. (2024). *Qwen2.5-1.5B-Instruct* [Model card]. [Hugging Face](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct). The repository's model card documents direct Transformers generation and the model context limit.
- ProsusAI. (n.d.). *FinBERT* [Model card]. [Hugging Face](https://huggingface.co/ProsusAI/finbert). Used for financial-text sentiment, not stock-return prediction.
- Aroussi, R. (n.d.). *yfinance documentation*. [Documentation](https://ranaroussi.github.io/yfinance/). Used for educational market/financial retrieval.
- Hays, T., Kaur, R., & Marx, A. (2026). *NLP_AAI520: Multi-Agent Financial Analysis System* [Source code and executed component notebooks]. [GitHub](https://github.com/tim17hays/NLP_AAI520).
- Course assignment. *AAI-520 Final Team Project Requirements*. Supplied project specification preserved in `docs/requirements/AAI-520_Final_Project_Requirements.pdf`.

The AAPL snapshot retains its original source URLs and access times. Controlled DEMO evidence is explicitly fictional. Historical controlled model results were extracted from Ashley's saved notebook stdout; no replacement scores or model responses were created during assembly.

## AI assistance disclosure and responsibility
Tim used Codex to assist with retrieval reliability, repository integration, synchronization of the authored functions, wording corrections, assembly, offline validation and export preparation. Roopleen's planning/Qwen code and Ashley's evaluator/revision/memory code and recorded experiments retain their authorship. Ashley's original notebook contains her own AI-use disclosure.

The team should review, understand and adapt this disclosure to reflect each member's actual tool use before submission. Local assembly did not perform new GPU inference. Fresh Colab runtime/model details and outputs are printed by this notebook when run there. No unverified team approval or submission claim is made.

In [14]:
import importlib.metadata as metadata
versions = {}
for package in ("torch", "transformers", "yfinance", "requests", "nbformat", "nbconvert"):
    try: versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError: versions[package] = "not installed in this audit runtime"
manifest = {"code_commit": TEAM_COMMIT, "finished_at": datetime.now(timezone.utc).isoformat(),
            "execution_mode": "fresh_colab_models" if RUN_LIVE_MODELS else "recorded_evidence_audit",
            "versions": versions, "workflow_checks": observed["workflow_checks"],
            "quality_checks": observed["quality_checks"], "manual_findings": manual_findings}
(OUTPUT_DIR / "submission_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
if IN_COLAB:
    from google.colab import _message, files
    import nbformat
    from scripts.export_submission import render_notebook
    from zipfile import ZipFile, ZIP_DEFLATED
    try:
        payload = _message.blocking_request("get_ipynb", timeout_sec=30)
        executed = nbformat.reads(json.dumps(payload["ipynb"]), as_version=4)
    except Exception as exc:
        print("Automatic notebook capture unavailable:", type(exc).__name__)
        print("Save the executed notebook with File → Download → .ipynb, then export it with scripts/export_submission.py.")
    else:
        # The current cell's output may not yet be included in Colab's snapshot.
        ipynb_path = OUTPUT_DIR / "final_multi_agent_financial_analysis.ipynb"
        nbformat.write(executed, ipynb_path)
        body = render_notebook(executed)
        ipynb_path.with_suffix(".html").write_text(body)
    archive = OUTPUT_DIR / "AAI520_final_submission_artifacts.zip"
    with ZipFile(archive, "w", ZIP_DEFLATED) as bundle:
        for p in OUTPUT_DIR.rglob("*"):
            if p.is_file() and p != archive:
                bundle.write(p, arcname=str(p.relative_to(OUTPUT_DIR)))
    print("Download bundle:", archive)
    files.download(str(archive))
else:
    print("Local audit complete. No new GPU results or Canvas submission are claimed.")

{
  "code_commit": "37f985826bdd388d473e3aa280c9c8e4b292fa06",
  "finished_at": "2026-10-19T20:44:01.477046+00:00",
  "execution_mode": "recorded_evidence_audit",
  "versions": {
    "torch": "not installed in this audit runtime",
    "transformers": "not installed in this audit runtime",
    "yfinance": "not installed in this audit runtime",
    "requests": "not installed in this audit runtime",
    "nbformat": "5.11.1",
    "nbconvert": "7.17.2"
  },
  "workflow_checks": {
    "run_1_initial_failure_detected": true,
    "run_1_revision_attempted": true,
    "run_1_changed_draft_evaluated": true,
    "run_1_process_lesson_saved": true,
    "run_1_lesson_read_from_disk": true,
    "run_2_retrieved_run_1_lesson": true,
    "run_2_plan_received_run_1_lesson": true,
    "run_2_generator_received_run_1_lesson": true,
    "all_workflow_checks_passed": true
  },
  "quality_checks": {
    "run_1_final_passed": false,
    "run_2_final_passed": false,
    "run_1_score_increased": false,
    "ru